### SCD type 0

These are those columns in a situation where you don't want to change the values of rows in these columns. Ex - employee_doj, employee_dob

### SCD type 1
**Merge(Upsert) Operation**

* Whenever new new data comes to source table we, need to update and insert latest record based on condition if matching values are present in source table and target table, then we need to update the record at target table and when there is no matching record then we need to simply insert these new records to target table.<br>This process is called **Merge/Upsert operation.**

* This is the most commonly used process used in any database development.
* This feature was not present in datalake but came with introduction of delta lake.
* Merge operation can be performed using spark SQL as well as pyspark but it is preferred to use pyspark methods
* Spark SQL has some limitation when it comes to merge operations

#### using SQL syntax

In [0]:
%fs ls 'user/hive/warehouse/'

In [0]:
spark.sql("""CREATE TABLE test_delta USING DELTA""")

In [0]:
def delta_check(TableName: str, location: str) -> bool:
    desc_table = spark.sql(f"describe formatted {TableName}").collect()
    location = [i[1] for i in desc_table if i[0] == 'Location'][0]
    try:
        dir_check = dbutils.fs.ls(f"{location}/_delta_log")
        is_delta = True
    except Exception as e:
        is_delta = False
    return is_delta

res = delta_check("test_delta", "dbfs:/user/hive/warehouse/test_delta/")

if (res=="True"):
    print("Yes, it is a Delta table.")
else:
    print("No, it is not a Delta table.")

In [0]:
%sql
SELECT * FROM delta.dbfs:/user/hive/warehouse/test_delta/

In [0]:
%sql
DROP TABLE IF EXISTS customer_source;
CREATE TABLE IF NOT EXISTS customer_source(id INT, customer_name STRING, location STRING, date DATE);
INSERT INTO customer_source VALUES (1,'Rohan','Bangalore', '2023-04-23');
INSERT INTO customer_source VALUES (4,'Sameer','Delhi', '2023-04-20');
INSERT INTO customer_source VALUES (5,'Zubin','Gurgaon', '2023-04-21');
SELECT * FROM customer_source;

In [0]:
%fs ls 'user/hive/warehouse/customer_source'

In [0]:
%sql
CREATE TABLE customer_source_test USING LOCATION 'user/hive/warehouse/customer_source/'

In [0]:
%run "/Delta Tables/Delta_table"

In [0]:
%sql
DROP TABLE customer_target;
CREATE TABLE IF NOT EXISTS customer_target(id INT, customer_name STRING, location STRING);
INSERT INTO customer_target VALUES (1,'Rohan','Gurgaon');
INSERT INTO customer_target VALUES (2,'Rahul','Chennai');
INSERT INTO customer_target VALUES (3,'Riya','Bangalore');

In [0]:
%sql
SELECT * FROM customer_source;

In [0]:
%sql
SELECT * FROM customer_target

In [0]:
%sql
MERGE INTO customer_target AS ct
USING customer_source AS cs
ON ct.id = cs.id
WHEN MATCHED THEN
UPDATE SET ct.location = cs.location
WHEN NOT MATCHED THEN
INSERT *

In [0]:
%sql
SELECT * FROM customer_target

#### using pyspark dataframe

In [0]:
from pyspark.sql.functions import StructType, StructField, StringType, IntegerType
schema = StrucType([])